## Logical flow
* per patient: load pt_neur_df, pt_trigs_df, pt_beh_df
    * pair trials with triggers: psychopy row i = i-th trigger trial
    * per epoch: extract spikes & FRs per trial & neuron, then stack patients along the neuron axis -> pseudopopulation
* save per epoch: spikes (trials, all_neurs), FRs (trials, all_neurs, bins; raw — norm downstream via flag), trial_mean_FRs (trials, all_neurs), bins (bins,)
* also save all_neur_df (neuron metadata: patient, region, stacked column idx) and per-patient trial tables

In [13]:
import pandas as pd, numpy as np, glob, os, pickle
from scipy.ndimage import gaussian_filter1d
from neo.io import BlackrockIO

In [14]:
pseudopop_dir = '../../outputs/processed_data/pseudopop'; os.makedirs(pseudopop_dir, exist_ok=True)
processed_dir = '../../outputs/processed_data'  # per-patient inputs (neurs_df.parquet)

all_beh_df = pd.read_csv('../../data/psychopy/all_subjs.csv')
patients = all_beh_df['subj'].unique(); patients = patients[patients > 11]; print(f'neural patients: {patients}')

size, dt = .02, .01 # smoothing params

# set epoch prestarts and durations (xlims)
epochs = ['baseline', 'stim', 'delay', 'response', 'feedback']; epoch_prestarts, epoch_durs = {}, {}

for epoch in epochs:

    # where to start x-axis
    if epoch == 'baseline': epoch_prestarts[epoch] = -0
    elif epoch == 'response': epoch_prestarts[epoch] = -1 # 1s before response submitted
    else: epoch_prestarts[epoch] = -.25

    # plot duration
    if epoch == 'delay': epoch_durs[epoch] = 1.5
    elif epoch == 'response': epoch_durs[epoch] = 0 # 0s after response submitted
    else: epoch_durs[epoch] = 1

print(f'prestarts: {epoch_prestarts}\ndurations: {epoch_durs}')

neural patients: [12. 18. 21. 22.]
prestarts: {'baseline': 0, 'stim': -0.25, 'delay': -0.25, 'response': -1, 'feedback': -0.25}
durations: {'baseline': 1, 'stim': 1, 'delay': 1.5, 'response': 0, 'feedback': 1}


### helpers

In [15]:
def get_pt_trigs(patient):
    ''' digital trigger stream from the nev file, times relative to block1 start '''
    nev_file = glob.glob(f'../../data/2025{int(patient)}/raw/*.nev')[0]; io = BlackrockIO(nev_file)
    seg = io.read_block(lazy=False).segments[0]; dig_ev = [ev for ev in seg.events if "digital" in ev.name.lower()][0]

    code_map = {10:"block started",20:"baseline started",30:"stim started",40:"delay started",50:"task started",51:"marker moved",52:"left pressed",53:"left released",54:"right pressed",55:"right released",56:"response submitted",60:"anticipation started",70:"feedback started",80:"block ended"}

    pt_trigs_df_raw = pd.DataFrame({"trigger_code": dig_ev.labels.astype(int),"time": dig_ev.times.magnitude}); pt_trigs_df_raw["event"] = pt_trigs_df_raw["trigger_code"].map(code_map)

    # align times relative to block1 start; drop triggers before it and abs time
    block1_start_idx = pt_trigs_df_raw.index[pt_trigs_df_raw["trigger_code"] == 10][0]; pt_trigs_df_raw["rel_time"] = pt_trigs_df_raw["time"] - pt_trigs_df_raw.loc[block1_start_idx, "time"]
    pt_trigs_df = pt_trigs_df_raw[block1_start_idx:]; pt_trigs_df = pt_trigs_df.drop(columns=['time']).reset_index(drop=True)
    return pt_trigs_df


def get_epoch_spikes_and_FRs(pt_trigs_df, neurs_df, epoch, size=size, dt=dt):
    ''' for each trial and neuron, get spike times and smoothed FRs (Hz) in epoch window
        pt_trigs_df: trig times, neurs_df: spike times per neur '''

    epoch_prestart, epoch_dur = epoch_prestarts[epoch], epoch_durs[epoch]
    bin_edges = np.arange(epoch_prestart, epoch_dur + dt, dt); bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2; n_bins = len(bin_edges) - 1

    # get epoch start indices & times
    if epoch != 'response': epoch_start_rows = pt_trigs_df[pt_trigs_df['event'] == f'{epoch} started'].index
    # using anticipation started instead of response submitted bc sometimes they dont respond
    else: epoch_start_rows = pt_trigs_df[pt_trigs_df['event'] == 'anticipation started'].index
    epoch_starts = pt_trigs_df.loc[epoch_start_rows, 'rel_time'].values; n_trials = len(epoch_starts)

    trial_neur_spikes, trial_neur_FRs = np.empty((n_trials, len(neurs_df)), dtype=object), np.zeros((n_trials, len(neurs_df), n_bins))

    for trial_idx in range(n_trials):
        for neur_idx, (_, neur_row) in enumerate(neurs_df.iterrows()):

            trial_epoch_spikes = neur_row['spikes'][(neur_row['spikes'] >= epoch_starts[trial_idx] + epoch_prestart) &
                                                    (neur_row['spikes'] <= epoch_starts[trial_idx] + epoch_dur)]
            trial_epoch_spikes = trial_epoch_spikes - epoch_starts[trial_idx] # align

            # bin and smooth
            counts, _ = np.histogram(trial_epoch_spikes, bins=bin_edges)
            smooth_spike_train = gaussian_filter1d(counts.astype(float), sigma=size/dt, mode='reflect', truncate=3.0)
            smooth_spike_train = smooth_spike_train / dt  # instantaneous FR (Hz): spikes/bin / s/bin

            trial_neur_spikes[trial_idx, neur_idx], trial_neur_FRs[trial_idx, neur_idx, :] = trial_epoch_spikes, smooth_spike_train

    return trial_neur_spikes, trial_neur_FRs, bin_centers


### per-patient extraction

In [16]:
pt_trial_tables, pt_neur_dfs, pt_epoch_data = {}, [], {}
for patient in patients:

    pt_neur_df = pd.read_parquet(f'{processed_dir}/2025{int(patient)}/neurs_df.parquet')
    pt_trigs_df = get_pt_trigs(patient)

    # trial rows currently in csv order; chronological sort ['run','blocks.thisN','trials.thisN'] lands here when the alignment fix is applied
    pt_beh_df = all_beh_df.loc[all_beh_df['subj'] == patient].reset_index(drop=True)
    pt_trial_tables[patient] = pt_beh_df

    pt_epoch_data[patient] = {epoch: get_epoch_spikes_and_FRs(pt_trigs_df, pt_neur_df, epoch) for epoch in epochs}
    pt_neur_dfs.append(pt_neur_df)
    pt_trigs_df.to_parquet(f'{processed_dir}/2025{int(patient)}/trigs_df.parquet')
    print(f'pt{int(patient)}: {len(pt_neur_df)} neurons, {len(pt_beh_df)} trials, {len(pt_trigs_df)} triggers')

pt12: 23 neurons, 240 trials, 2454 triggers
pt18: 13 neurons, 240 trials, 2546 triggers
pt21: 14 neurons, 240 trials, 2449 triggers
pt22: 7 neurons, 240 trials, 2344 triggers


### stack patients along the neuron axis -> pseudopopulation

In [ ]:
# neuron metadata: which stacked column is which neuron
all_neur_df = pd.concat(pt_neur_dfs, ignore_index=True)
all_neur_df['pop_neur_idx'] = np.concatenate([np.arange(len(df)) for df in pt_neur_dfs])  # column within own patient
all_neur_df['col_idx'] = np.arange(len(all_neur_df))                                      # column in stacked matrices
all_neur_df.to_parquet(f'{pseudopop_dir}/all_neur_df.parquet')
print(f'all_neur_df: {len(all_neur_df)} neurons across {all_neur_df["patient"].nunique()} patients')
all_neur_df[['patient', 'chanID', 'unitID', 'region', 'pop_neur_idx', 'col_idx']].head()

In [18]:
for epoch in epochs:
    os.makedirs(f'{pseudopop_dir}/{epoch}', exist_ok=True)
    spikes = np.hstack([pt_epoch_data[pt][epoch][0] for pt in patients])                  # (trials, all_neurs) spike times
    FRs    = np.concatenate([pt_epoch_data[pt][epoch][1] for pt in patients], axis=1)     # (trials, all_neurs, bins)
    bins   = pt_epoch_data[patients[0]][epoch][2]
    trial_mean_FRs = FRs.mean(axis=2)  # scalar FR per (trial, neuron); feeds tuning heatmaps & decoding

    np.save(f'{pseudopop_dir}/{epoch}/spikes.npy', spikes, allow_pickle=True)
    np.save(f'{pseudopop_dir}/{epoch}/FRs.npy', FRs, allow_pickle=True)
    np.save(f'{pseudopop_dir}/{epoch}/trial_mean_FRs.npy', trial_mean_FRs, allow_pickle=True)
    np.save(f'{pseudopop_dir}/{epoch}/bin_centers.npy', bins, allow_pickle=True)
    print(f'{epoch}: spikes {spikes.shape}, FRs {FRs.shape}, trial_mean_FRs {trial_mean_FRs.shape}, bins {bins.shape}')

with open(f'{pseudopop_dir}/trial_tables.pkl', 'wb') as f: pickle.dump(pt_trial_tables, f)
print('saved trial_tables.pkl')

baseline: spikes (240, 57), FRs (240, 57, 100), trial_mean_FRs (240, 57), bins (100,)
stim: spikes (240, 57), FRs (240, 57, 125), trial_mean_FRs (240, 57), bins (125,)
delay: spikes (240, 57), FRs (240, 57, 175), trial_mean_FRs (240, 57), bins (175,)
response: spikes (240, 57), FRs (240, 57, 100), trial_mean_FRs (240, 57), bins (100,)
feedback: spikes (240, 57), FRs (240, 57, 125), trial_mean_FRs (240, 57), bins (125,)
saved trial_tables.pkl
